# Chapter 09 — Retrieval Evaluation

*Where we are:* we can retrieve — but **is it any good?** Without ground truth and metrics, every
"improvement" is a guess.

```
retrievers →[ labelled benchmark + P@k / R@k / MRR / NDCG ]→ measured comparison
```

In [1]:
# === Chapter 09 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 09 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 09 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 37. Ground truth

We hand-label a small **patent-query benchmark**: each query has gold relevant documents, a
grounded reference answer, and reference evidence. The queries are deliberately **low-overlap
paraphrases** of the patents' wording — a realistic patent-search situation where the searcher
does not know the patentee's exact terms.

In [2]:
import pandas as pd
from patentrag.evaluation import build_eval_dataset
ds = bs.ensure("eval_dataset")   # also materializes the artifact for later chapters
print(f"{len(ds)} labelled queries\n")
pd.DataFrame([{"query": e.query[:60], "gold_docs": ", ".join(e.relevant_doc_ids),
               "evidence": f"{e.reference_evidence[0]}/{e.reference_evidence[1]}"} for e in ds[:6]])

12 labelled queries



,query,gold_docs,evidence
0,train vector representations so related media ...,US11971885B2,US11971885B2/abstract
1,compactly keep millions of feature vectors so ...,"US11093561B2, US11714853B2",US11714853B2/abstract
2,suggest products to a shopper in real time fro...,US11113744B2,US11113744B2/abstract
3,match a meaning vector against documents acros...,US11216459B2,US11216459B2/abstract
4,figure out which of several possible meanings ...,US9183310B2,US9183310B2/abstract
5,decide which fragments of a document should go...,US9342582B2,US9342582B2/abstract


## 38–41. The metrics, derived and implemented

For a ranked list and a relevant set $R$:

$$P@k=\frac{|R\cap \text{top-}k|}{k}\qquad R@k=\frac{|R\cap \text{top-}k|}{|R|}\qquad \text{RR}=\frac{1}{\text{rank of first relevant}}\qquad \text{MRR}=\frac1{|Q|}\sum_q \text{RR}_q$$

$$\text{DCG}@k=\sum_{i=1}^{k}\frac{rel_i}{\log_2(i+1)}\qquad \text{NDCG}@k=\frac{\text{DCG}@k}{\text{IDCG}@k}$$

**Precision** = fraction of returned that are relevant; **Recall** = fraction of relevant that
were returned (prior-art search prizes recall). **MRR** rewards ranking the first hit high.
**NDCG** rewards graded relevance discounted by rank. All implemented from these formulas in
`patentrag/evaluation.py`; a worked example:

In [3]:
from patentrag.evaluation import precision_at_k, recall_at_k, reciprocal_rank, ndcg_at_k
ranked = ["A", "B", "C", "D", "E"]      # B, D are the relevant ones
rel = {"B", "D"}
print(f"P@3 = {precision_at_k(ranked, rel, 3):.3f}   (1 of top-3 relevant)")
print(f"R@5 = {recall_at_k(ranked, rel, 5):.3f}   (both relevant recovered)")
print(f"RR  = {reciprocal_rank(ranked, rel):.3f}   (first relevant at rank 2)")
print(f"NDCG@5 = {ndcg_at_k(ranked, {'B':1.0,'D':1.0}, 5):.3f}")

P@3 = 0.333   (1 of top-3 relevant)
R@5 = 1.000   (both relevant recovered)
RR  = 0.500   (first relevant at rank 2)
NDCG@5 = 0.651


## 42. The retrieval experiment

We compare **BM25 / Dense / Hybrid (RRF) / Hybrid + cross-encoder** on the benchmark, evaluating
at **document granularity** (a chunk hit credits its document). Models are warmed up first so the
latency column reflects steady-state, not one-time model loading.

In [4]:
import time
from patentrag.dense import DenseRetriever
from patentrag.fusion import reciprocal_rank_fusion, CrossEncoderReranker
from patentrag.evaluation import evaluate_retriever

chunks = bs.ensure("chunks"); by_id = {c.chunk_id: c for c in chunks}
bm25 = bs.ensure("bm25_index"); emb = bs.ensure("embeddings")
dense = DenseRetriever(emb["chunk_ids"], emb["matrix"])
reranker = CrossEncoderReranker()

def hybrid(q, k):
    b = [c for c, _ in bm25.search(q, 50)]; d = [c for c, _ in dense.search(q, 50)]
    return [(cid, s) for cid, s in reciprocal_rank_fusion([b, d])][:k]

def hybrid_rerank(q, k):
    cand = hybrid(q, 30)
    return reranker.rerank(q, [(cid, by_id[cid].for_index()) for cid in [c for c, _ in cand]], top_k=k)

# warm up model paths (exclude cold-load from latency)
dense.search("warm up", 5); reranker.rerank("warm", [("x", "warm up passage")])

def timed(fn):
    t0 = time.perf_counter(); m = evaluate_retriever(fn, ds, by_id); return m, (time.perf_counter()-t0)/len(ds)*1000

rows = []
for name, fn in [("BM25", bm25.search), ("Dense", dense.search), ("Hybrid (RRF)", hybrid),
                 ("Hybrid + reranker", hybrid_rerank)]:
    m, ms = timed(fn)
    rows.append({"retriever": name, "R@5": round(m["recall@5"], 2), "R@10": round(m["recall@10"], 2),
                 "MRR": round(m["mrr"], 2), "NDCG@10": round(m["ndcg@10"], 2), "ms/query": round(ms, 1)})
retrieval_metrics = pd.DataFrame(rows)
bs.save_artifact("retrieval_metrics", retrieval_metrics.to_dict("records"))
retrieval_metrics

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8064.16it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8985.96it/s]

,retriever,R@5,R@10,MRR,NDCG@10,ms/query
0,BM25,0.96,1.0,0.59,0.69,2.2
1,Dense,1.00,1.0,0.96,0.97,6.2
2,Hybrid (RRF),1.00,1.0,0.96,0.96,9.1
3,Hybrid + reranker,1.00,1.0,1.00,0.99,630.0


### Interpretation

- **BM25** recalls the right documents (small corpus) but its **MRR/NDCG lag**: paraphrased
  queries don't share the patentee's exact words, so the right passage isn't ranked first.
- **Dense** repairs vocabulary mismatch — big MRR/NDCG jump — at higher latency (embedding + full
  scan).
- **Hybrid (RRF)** is at least as good as its best arm, robustly, for negligible cost.
- **+ Reranker** pushes the correct passage to the top (best MRR/NDCG) at the largest latency
  cost — the classic precision/latency trade.

### Concept distinction (SPEC §18)

Do **not** conflate these — they are measured differently and fail independently:

| Term | Question it answers | Measured by |
|---|---|---|
| **Retrieval relevance** | Did we fetch the right *documents*? | R@k, MRR, NDCG (this chapter) |
| **Answer relevance** | Is the generated *answer* on-topic for the query? | Ch 10 (embedding sim) |
| **Faithfulness / groundedness** | Is the answer *supported by* the retrieved context? | Ch 10 / Ch 12 |
| **Citation correctness** | Do the answer's citations resolve to real support? | Ch 12 |

A perfect retriever can still yield a wrong answer (bad generation), and a fluent answer can be
unfaithful. Later chapters measure each separately.

## Chapter invariants

In [5]:
m = retrieval_metrics.set_index("retriever")
assert (m["R@5"] <= 1.0).all() and (m["NDCG@10"] <= 1.0).all()
# semantic retrieval improves ranking quality over pure lexical on these paraphrased queries
assert m.loc["Dense", "MRR"] >= m.loc["BM25", "MRR"]
assert m.loc["Hybrid + reranker", "NDCG@10"] >= m.loc["Hybrid (RRF)", "NDCG@10"]
assert abs(precision_at_k(ranked, rel, 3) - 1/3) < 1e-9   # manual metric matches the definition
print("All Chapter 09 invariants hold. eval_dataset + retrieval_metrics artifacts ready.")

All Chapter 09 invariants hold. eval_dataset + retrieval_metrics artifacts ready.


In [6]:
# === Chapter 09 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['sentence-transformers', 'rank-bm25', 'pandas']
print("Chapter 09 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 09 VALIDATION: PASS")

Chapter 09 — environment
  Python : 3.12.10 on Windows 11
  sentence-transformers   : 6.0.0
  rank-bm25               : 0.2.2
  pandas                  : 3.0.2

CHAPTER 09 VALIDATION: PASS
